# Reproducing BERT4Rec Results on ML1M

This notebook reproduces the MovieLens 1M data preparation and evaluation
protocols for Compresso RecSys's **BERT4Rec**. Unlike the SASRec notebook in
this repository, the model here is a faithful port rather than a modernized
variant: post-norm transformer blocks, the reference's LayerNorm placements in
the embedding and the output head, tied item embeddings, and the Cloze
objective as published. `Bert4RecConfig` records a provenance per field.

The trained model is evaluated under two protocols:

1. the paper's held-out target against 100 sampled negatives at cutoff 10;
2. Compresso RecSys's full-catalog protocol at cutoff 20.

Keeping the protocols separate matters: sampled ranking is much easier than
ranking the entire catalog, so their numbers are not interchangeable.

**The sampled protocol here is not the one in the SASRec notebook.** §4.2 of
the BERT4Rec paper samples its 100 negatives "according to their popularity",
where SASRec samples uniformly from the unseen items. Popularity sampling
draws harder negatives, so it produces lower numbers on the same model, and
the two are not comparable even at the same cutoff. Using SASRec's sampler
here would silently flatter the result.

## Configuration and run modes

`Bert4RecConfig`'s defaults are the published settings, but they follow the
ml-20m scripts where the four `run_*.sh` disagree. Two changes align them with
ml-1m: `max_predictions` is 40 rather than 20 (`max_predictions_per_seq` in
`run_ml-1m.sh`), and `epochs` is computed below.

**`epochs` has no published counterpart and the default is far too small for
this.** The reference trains a fixed `num_train_steps` of 400000 over a corpus
generated once; here the step count is `epochs * samples / batch_size`, so the
equivalent epoch budget depends on how many Cloze samples ML-1M produces and
has to be derived. It lands in the high hundreds to low thousands. That is not
a mistake in the arithmetic -- BERT4Rec is known to need far longer training
than sequential recommenders of its era, and replication studies attribute much
of the gap between reported and reproduced numbers to stopping too early.

Expect hours on a GPU and do not attempt the full budget on CPU: the run
recorded at the end of this notebook derived 907 epochs and spent about six
and a half hours on them. Set `SMOKE_EPOCHS` to a small integer to check the
workflow first. Smoke runs deliberately hide the published comparison column
so a shortened run cannot be mistaken for a full experiment.

In [ ]:
from dataclasses import replace
from pathlib import Path
import time

import numpy as np
import torch
from compresso import SRPTensor

import compresso_recsys as cr
from compresso_recsys.evaluation import (
    evaluate_ranked_predictions,
    evaluate_recommender,
)
from compresso_recsys.metrics import MRR, NDCG, HitRate, Recall
from compresso_recsys.models import (
    Bert4RecConfig,
    Bert4RecTrainer,
    ItemTokenizer,
    SequenceBatcher,
)
from compresso_recsys.models.bert4rec.trainer import SPECIAL_TOKENS
from compresso_recsys.sequences import ItemSequences

# Sun et al., CIKM 2019, Table 2, ML-1m row.
PUBLISHED_HR10 = 0.6970
PUBLISHED_NDCG10 = 0.4818
N_NEGATIVES = 100
SAMPLED_CUTOFF = 10
NEGATIVE_SEED = 0
FULL_CUTOFF = 20
# run_ml-1m.sh: --num_train_steps=400000
TARGET_STEPS = 400_000
CHECKPOINT = Path("artifacts/ml1m/bert4rec_ml1m.zip")

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

SMOKE_EPOCHS: int | None = None  # Set to 1 for a quick workflow check.

# max_predictions 40 is run_ml-1m.sh's max_predictions_per_seq; the config
# default of 20 is ml-20m's. Everything else already matches ml-1m.
cfg = replace(Bert4RecConfig(), device=DEVICE, max_predictions=40)

started = time.time()

def elapsed() -> str:
    return f"[{time.time() - started:7.1f}s]"

print(
    f"device: {DEVICE}   window: {cfg.max_history_length}   "
    f"rho: {cfg.mask_proportion}   max_predictions: {cfg.max_predictions}"
)
print("model: Compresso BERT4Rec (faithful port; see Bert4RecConfig docstring)")
if SMOKE_EPOCHS is not None:
    print("SMOKE_EPOCHS is set: this is a smoke run, not a full experiment")

## Build the MovieLens 1M checkpoint

The paper uses chronological leave-last-out evaluation and treats every rating
as implicit feedback, keeping users and items with at least five interactions.
`min_value_to_keep=1.0` therefore keeps all ratings, and entity-text filtering
is disabled since BERT4Rec reads item IDs rather than text.

These are the same build parameters as the SASRec notebook. The validation
checklist on the *Dataset validation* page asks that competing models reuse
one identical checkpoint, so point `CHECKPOINT` at
`artifacts/ml1m/sasrec_ml1m.zip` if you have already built it.

In [ ]:
print(f"{elapsed()} checkpoint")
if CHECKPOINT.exists():
    print(f"reusing {CHECKPOINT}")
else:
    print(f"building {CHECKPOINT} from MovieLens 1M")
    cr.build_recsys_checkpoint(
        dataset="ml1m",
        data_dir="data",
        checkpoint_path=str(CHECKPOINT),
        split_mode="leave_last_out",
        min_user_support=5,
        item_min_support=5,
        min_value_to_keep=1.0,
        min_entity_text_words=0,
        seed=0,
        show_progress=False,
    )

## Inspect the split

Leave-last-out reserves each user's last interaction for test and the
second-to-last for validation, which is §4.2's split. The paper's dataset
statistics describe the whole sequence, so the prepared training split is two
interactions per user shorter.

In [ ]:
with cr.read_checkpoint(CHECKPOINT) as root:
    split = cr.load_recsys_split(root)

train_sequences = split["x_train_sequences"]
test_source = split["test_source_sequences"]
test_targets = split["test_target_matrix"]
item_ids = split["train_item_ids"]

lengths = np.diff(train_sequences.indptr)
users = train_sequences.n_rows
items = train_sequences.n_items
actions = int(train_sequences.values.size)
print(f"{users} users, {items} items, {actions} actions")
print(
    f"{actions / users:.1f} actions/user, "
    f"{actions / items:.1f} actions/item, "
    f"median history {int(np.median(lengths))}"
)
print("paper Table 1: 6040 users, 3416 items, 163.5 avg length, 4.79% density")
print(
    f"histories longer than the window ({cfg.max_history_length}): "
    f"{int((lengths > cfg.max_history_length).sum())}"
)

## Derive the epoch budget from the reference's step count

One history becomes many Cloze samples: sliding windows over anything longer
than `max_history_length`, `duplication_factor` maskings of each window, and
one last-item-masked sample per window. `_training_windows` is the function
`fit` itself calls, so counting with it gives the exact sample count the run
will use.

`epochs` is then whatever makes the step count match `run_ml-1m.sh`'s 400000.
Print it before committing to it.

In [ ]:
probe = Bert4RecTrainer(cfg)
n_samples = len(probe._training_windows(train_sequences))
steps_per_epoch = -(-n_samples // cfg.batch_size)
epochs_for_target = max(1, round(TARGET_STEPS / steps_per_epoch))

print(f"{n_samples} Cloze samples per epoch ({steps_per_epoch} steps)")
print(f"{TARGET_STEPS} reference steps -> {epochs_for_target} epochs here")

if SMOKE_EPOCHS is not None:
    cfg = replace(cfg, epochs=SMOKE_EPOCHS)
    print(f"SMOKE_EPOCHS overrides this: running {cfg.epochs} epoch(s)")
else:
    cfg = replace(cfg, epochs=epochs_for_target)
    print(
        f"running {cfg.epochs} epochs = "
        f"{cfg.epochs * steps_per_epoch} steps; budget hours, not minutes"
    )

## Train BERT4Rec

The tokenizer needs the Cloze special tokens -- `[mask]` above all, since the
objective is not expressible without one -- and retains MovieLens item IDs for
stable-ID recommendation. The batcher leaves its window unset so
`Bert4RecConfig.max_history_length` remains the single source of truth; `fit`
also forces right padding, which is what a learned positional table reads under
Cloze.

In [ ]:
print(f"{elapsed()} train")
batcher = SequenceBatcher(
    ItemTokenizer(
        train_sequences.n_items,
        item_ids=item_ids,
        special_tokens=SPECIAL_TOKENS,
    ),
)
trainer = Bert4RecTrainer(cfg, batcher).fit(
    train_sequences,
    item_ids=item_ids,
)

every = max(1, len(trainer.history) // 10)
print(f"{'epoch':>6}  {'loss':>9}")
for index, entry in enumerate(trainer.history):
    if index % every == 0 or index == len(trainer.history) - 1:
        print(f"{int(entry['epoch']):>6}  {entry['loss']:>9.4f}")

## Score the catalog

The sampled protocol needs scores for specific items, including candidates
outside the model's top-k result. Compute the full score matrix once in batches
and reuse it below.

This walks the same path `predict_on_batch` does -- append `[mask]` to each
history, read the hidden state at that position, drop the reserved columns --
without the top-k and the seen-item mask, which the protocols below apply
themselves.

In [ ]:
def catalog_scores(
    model: Bert4RecTrainer,
    source: ItemSequences,
    *,
    rows: int = 512,
) -> np.ndarray:
    out = np.empty((source.n_rows, source.n_items), dtype=np.float32)
    model.model.eval()
    for start in range(0, source.n_rows, rows):
        stop = min(start + rows, source.n_rows)
        chunk = ItemSequences(
            values=source.values[source.indptr[start] : source.indptr[stop]],
            indptr=source.indptr[start : stop + 1] - source.indptr[start],
            n_items=source.n_items,
        )
        with torch.no_grad():
            tokens, padding = model._encode_with_mask(chunk)
            logits = model.model(tokens, padding)
            out[start:stop] = (
                logits[:, model._item_offset :].float().cpu().numpy()
            )
    return out

print(f"{elapsed()} scoring {test_source.n_rows} test users")
scores = catalog_scores(trainer, test_source)

## Published protocol: 100 popularity-sampled negatives at 10

For each user, rank the held-out item against 100 items that user never
interacted with, drawn with probability proportional to how often each item
appears in training. Ties go to the negative; otherwise an untrained model that
assigns every item the same score would report a perfect hit rate.

Two details the paper leaves open, resolved here and worth recording with any
result, because both move the numbers:

* **Weighting.** "According to their popularity" is read as proportional to
  training interaction count. A squared or log-damped weighting is an equally
  literal reading and a different benchmark.
* **Replacement.** Sampling is without replacement, so the 100 negatives are
  distinct. With replacement, a duplicate negative wastes a slot and the
  protocol gets slightly easier.

Items with zero training interactions cannot be popularity-sampled at all and
are excluded from the pool rather than given a floor.

In [ ]:
def sampled_protocol(
    scores: np.ndarray,
    interacted: np.ndarray,
    targets,
    popularity: np.ndarray,
    *,
    n_negatives: int,
    cutoff: int,
    seed: int,
) -> tuple[float, float, int]:
    rng = np.random.default_rng(seed)
    hits: list[float] = []
    gains: list[float] = []
    for row in range(scores.shape[0]):
        positives = targets[row].indices
        if positives.size == 0:
            continue
        pool = np.flatnonzero(~interacted[row] & (popularity > 0))
        if pool.size < n_negatives:
            continue
        weights = popularity[pool]
        negatives = rng.choice(
            pool,
            size=n_negatives,
            replace=False,
            p=weights / weights.sum(),
        )
        target = int(positives[0])
        above = int((scores[row, negatives] >= scores[row, target]).sum())
        hits.append(float(above < cutoff))
        gains.append(1.0 / np.log2(above + 2) if above < cutoff else 0.0)
    return float(np.mean(hits)), float(np.mean(gains)), len(hits)

popularity = np.asarray(split["x_train"].sum(axis=0)).ravel()
interacted = (
    split["test_source_matrix"] + test_targets
).astype(bool).toarray()
hr, ndcg, scored = sampled_protocol(
    scores,
    interacted,
    test_targets,
    popularity,
    n_negatives=N_NEGATIVES,
    cutoff=SAMPLED_CUTOFF,
    seed=NEGATIVE_SEED,
)

print(
    f"{elapsed()} published sampled protocol -- "
    f"{N_NEGATIVES} popularity-sampled negatives, @{SAMPLED_CUTOFF}"
)
print(f"{'metric':<12}{'this run':>10}{'published*':>12}{'delta':>10}")
if SMOKE_EPOCHS is not None:
    print(f"{'HR@10':<12}{hr:>10.4f}{'--':>12}{'--':>10}")
    print(f"{'NDCG@10':<12}{ndcg:>10.4f}{'--':>12}{'--':>10}")
    print("published column withheld: SMOKE_EPOCHS shortened this run")
else:
    print(
        f"{'HR@10':<12}{hr:>10.4f}{PUBLISHED_HR10:>12.4f}"
        f"{hr - PUBLISHED_HR10:>+10.4f}"
    )
    print(
        f"{'NDCG@10':<12}{ndcg:>10.4f}{PUBLISHED_NDCG10:>12.4f}"
        f"{ndcg - PUBLISHED_NDCG10:>+10.4f}"
    )
print("* BERT4Rec Table 2, ML-1m row. Sampling details above are this notebook's.")
print(f"scored {scored} of {test_source.n_rows} users")

## Package protocol: full catalog at 20

Now rank every unseen catalog item and compare BERT4Rec with a most-popular
baseline evaluated identically. This is the protocol used by the package's
benchmark tables, and the numbers are much lower than the sampled ones by
construction: ranking against 3415 candidates is not ranking against 100.

In [ ]:
metrics = [
    NDCG(FULL_CUTOFF),
    Recall(FULL_CUTOFF),
    HitRate(FULL_CUTOFF),
    MRR(FULL_CUTOFF),
]
result = evaluate_recommender(
    trainer,
    source=test_source,
    targets=test_targets,
    metrics=metrics,
    sample_ids=split["test_eval_user_ids"],
)

by_popularity = np.argsort(-popularity, kind="stable")
seen = split["test_source_matrix"].astype(bool).toarray()[:, by_popularity]
unseen_first = np.argsort(seen, axis=1, kind="stable")[:, :FULL_CUTOFF]
baseline = evaluate_ranked_predictions(
    predictions=SRPTensor(
        cols=torch.from_numpy(by_popularity[unseen_first]).long(),
        vals=torch.arange(
            FULL_CUTOFF,
            0,
            -1,
            dtype=torch.float32,
        ).expand(test_source.n_rows, FULL_CUTOFF),
        shape=(test_source.n_rows, train_sequences.n_items),
    ),
    targets=test_targets,
    metrics=metrics,
    sample_ids=split["test_eval_user_ids"],
)

print(f"{elapsed()} benchmark protocol -- full catalog, @{FULL_CUTOFF}")
print(f"{'metric':<16}{'BERT4Rec':>10}{'popular':>10}{'lift':>9}")
for name in result.metrics:
    ours, theirs = result[name], baseline[name]
    lift = f"{ours / theirs:.1f}x" if theirs else "--"
    print(f"{name:<16}{ours:>10.4f}{theirs:>10.4f}{lift:>9}")
print(f"scored {result.n_scored_rows} of {result.n_rows} users")

## Check the epoch budget on validation data

The budget above is derived from a step count, not tuned, so it is worth
checking. Validation and test nDCG tracking each other suggests it is
reasonable; validation far above test is a warning that the run overfit, and
both far below a shorter run is a warning it was trained past its peak.

In [ ]:
validation = evaluate_recommender(
    trainer,
    source=split["val_source_sequences"],
    targets=split["val_target_matrix"],
    metrics=[NDCG(FULL_CUTOFF)],
    sample_ids=split["val_eval_user_ids"],
)
val_ndcg = validation[f"ndcg@{FULL_CUTOFF}"]
test_ndcg = result[f"ndcg@{FULL_CUTOFF}"]
print(
    f"validation ndcg@{FULL_CUTOFF}: {val_ndcg:.4f}   "
    f"test ndcg@{FULL_CUTOFF}: {test_ndcg:.4f}"
)

## Save the trained model

Store the fitted recommender under `models/bert4rec.zip` inside the same data
checkpoint. The model entry carries its configuration, tokenizer, stable item
IDs, training history, and learned Torch state.

In [ ]:
trainer.save_to_checkpoint(CHECKPOINT, "bert4rec")
print(f"{elapsed()} saved into {CHECKPOINT.name} as 'bert4rec'")

## The recorded run

The tables below record one complete run. Like the SASRec notebook, this one
ships with its cells cleared: `nbsphinx_execute` is `never`, so anything
committed in an output cell is published as documentation and stays published,
whether it came from a finished run, a smoke run or an interrupted one. The
result belongs somewhere it is stated deliberately -- here, and in the
*Dataset validation* page's BERT4Rec / ML-1M row. A number without its
provenance is not a result, so:

| field | value |
|---|---|
| commit | `261dd51`, working tree clean apart from this notebook |
| device | one CUDA GPU |
| training | 907 epochs = 399987 steps, 23718s between the `train` and `scoring` prints |
| model | `Bert4RecConfig()` with `max_predictions=40`; seed 0, `lr` 1e-4, `batch_size` 256, rho 0.2, window 200, `duplication_factor` 10 |
| checkpoint | `ml1m` `leave_last_out`, build seed 0 -- 6040 users, 3416 items, 987531 training actions |
| negatives | seed 0, 100 popularity-weighted draws without replacement |

| protocol | this run | reference |
|---|---|---|
| 100 popularity-sampled negatives @10 | HR 0.6965, NDCG 0.4805 | paper Table 2: 0.6970, 0.4818 |
| full catalog @20 | nDCG 0.1954, Recall 0.4146, MRR 0.1336 | most-popular: 0.0258, 0.0682, 0.0145 |

Both protocols scored all 6040 test users. Validation nDCG@20 was 0.2011
against 0.1954 on test, the ordering a budget that has not overfit produces,
and the training loss was flat over its last hundred epochs -- 5.2389 at 811
and 5.2373 at 907 -- so the derived budget is long enough rather than merely
long.

**This stays labelled `adapted benchmark`, and the size of the deltas is not
why.** Point 6 of that page's checklist asks preprocessing, architecture,
training and evaluation to all match first. The first two do. The third is a
budget derived from the reference's 400000 steps, not a published epoch count,
and the fourth resolves "according to their popularity" as interaction-count
weighting without replacement, which §4.2 does not state. Either reading could
be wrong in a way this agreement would not reveal.

One seed also cannot separate a 0.0005 gap from a 0.005 one, so these
particular deltas are weaker evidence than their size suggests; they say the
implementation is in the right place, not that it is exact. Repeating the run
across seeds is the next thing this row needs.